In [2]:
# --- Cell 1: 强制用本地仓库版本（可选，但建议） ---
import sys, importlib
sys.path.insert(0, "/home/sh3ng/projects/GWSpace")
import gwspace.fishertool as ft
importlib.reload(ft)

<module 'gwspace.fishertool' from '/home/sh3ng/anaconda3/envs/gwspace312/lib/python3.12/site-packages/gwspace/fishertool.py'>

In [ ]:
# --- Cell 2: 基本导入 ---
import numpy as np
from gwspace.Waveform import EMRIWaveform
from gwspace.response import get_AET_td
from gwspace.Noise import TianQinNoise
from gwspace.constants import YRSID_SI
from gwspace.fishertool import fisher_matrix
# --- Cell 3: EMRI 参数 ---
EMRIpars = {
    "M": 4.3e6,
    "mu": 40.0,
    "a": 0.1,
    "p0": 9.0,
    "e0": 0.0,
    "x0": 1.0,
    "qS": 1.66,
    "phiS": 4.71,
    "qK": 0.2,
    "phiK": 0.2,
    "dist": 8.0e-6,
    "Phi_phi0": 1.0,
    "Phi_theta0": 2.0,
    "Phi_r0": 3.0,
    "T_obs": 5 * YRSID_SI,
}


In [13]:
# --- Cell 4: EMRIWaveform 实例 ---
wf = EMRIWaveform(**EMRIpars)

In [7]:
# --- Cell 5: EMRI -> AET 频域 wrapper（带参数转发） ---
class EMRIAETAdapter:
    _local = {"wf", "dt", "det", "TDIgen"}

    def __init__(self, wf, dt, det="TQ", TDIgen=1):
        object.__setattr__(self, "wf", wf)
        object.__setattr__(self, "dt", dt)
        object.__setattr__(self, "det", det)
        object.__setattr__(self, "TDIgen", TDIgen)

    def __getattr__(self, name):
        return getattr(self.wf, name)

    def __setattr__(self, name, value):
        if name in self._local:
            object.__setattr__(self, name, value)
        else:
            setattr(self.wf, name, value)

    def get_tdi_response(self, f_series=None, channel="AET", det=None, TDIgen=None, **kwargs):
        if channel.upper() != "AET":
            raise ValueError("EMRIAETAdapter 只支持 AET 通道。")
        det = det or self.det
        TDIgen = TDIgen or self.TDIgen

        t = np.arange(0, self.wf.T_obs, self.dt)
        A, E, T = get_AET_td(self.wf, t, det=det, TDIgen=TDIgen)

        N = len(t)
        f_fft = np.fft.rfftfreq(N, self.dt)
        A_f = np.fft.rfft(A) * self.dt
        E_f = np.fft.rfft(E) * self.dt
        T_f = np.fft.rfft(T) * self.dt
        h = np.vstack([A_f, E_f, T_f])

        if f_series is None:
            return h

        f_series = np.asarray(f_series)
        if f_series.shape == f_fft.shape and np.allclose(f_series, f_fft):
            return h

        def interp_complex(y):
            return np.interp(f_series, f_fft, y.real) + 1j * np.interp(f_series, f_fft, y.imag)

        return np.vstack([interp_complex(A_f), interp_complex(E_f), interp_complex(T_f)])


In [ ]:
# --- Cell 6: 频率轴与 FIM ---
dt = 100.0  # 秒，先用粗一点调试
adapter = EMRIAETAdapter(wf, dt=dt, det="TQ", TDIgen=1)

# 和 FFT 完全一致的频率轴，避免插值
t = np.arange(0, wf.T_obs, dt)
f_series = np.fft.rfftfreq(len(t), dt)
# f_series = f_series[1:]   # 去掉 DC

# 参数列表（psi/iota 对 EMRIWaveform 不生效，建议先别加）
params = [
    "M","mu","p0"
]

result = fisher_matrix(
    adapter,
    params=params,
    det="TQ",
    channel="AET",
    TDIgen=1,
    f_series=f_series,
    noise=TianQinNoise(),
    use_T=False,
    rel_step=1e-6,
)

print("SNR:", result["snr"])
print("sigma:", result["sigma"])


/home/sh3ng/anaconda3/envs/gwspace312/lib/python3.12/site-packages/gwspace/Noise.py:154: RuntimeWarning: divide by zero encountered in divide
  Sa_a = self.Na * (1. + 1e-4/freq)
/home/sh3ng/anaconda3/envs/gwspace312/lib/python3.12/site-packages/gwspace/Noise.py:43: RuntimeWarning: invalid value encountered in multiply
  Sa_nu = Sa_d*(2*PI*freq/C_SI)**2


SNR: 1082898728.5301068
sigma: {'M': 1.1224452053254948e-19, 'mu': 6.852679398176441e-14, 'p0': 6.621867324749536e-15}


In [17]:
import numpy as np
from gwspace.Waveform import EMRIWaveform
from gwspace.response import get_AET_td
from gwspace.Noise import TianQinNoise
from gwspace.constants import YRSID_SI

# ====== EMRI 参数 ======
EMRIpars = dict(
    M=4.3e6, mu=40.0, a=0.1, p0=9.0, e0=0.0, x0=1.0,
    qS=1.66, phiS=4.71, qK=0.2, phiK=0.2,
    dist=8.0e-6,
    Phi_phi0=1.0, Phi_theta0=2.0, Phi_r0=3.0,
    T_obs=5*YRSID_SI
)
wf = EMRIWaveform(**EMRIpars)

# ====== Adapter (时域 -> 频域 AET) ======
class EMRIAETAdapter:
    _local = {"wf", "dt", "det", "TDIgen"}
    def __init__(self, wf, dt, det="TQ", TDIgen=1):
        object.__setattr__(self, "wf", wf)
        object.__setattr__(self, "dt", dt)
        object.__setattr__(self, "det", det)
        object.__setattr__(self, "TDIgen", TDIgen)

    def __getattr__(self, name):
        return getattr(self.wf, name)

    def __setattr__(self, name, value):
        if name in self._local:
            object.__setattr__(self, name, value)
        else:
            setattr(self.wf, name, value)

    def get_fd_AET(self, f_series=None):
        t = np.arange(0, self.wf.T_obs, self.dt)
        A, E, T = get_AET_td(self.wf, t, det="TQ", TDIgen=1)
        N = len(t)
        f_fft = np.fft.rfftfreq(N, self.dt)
        A_f = np.fft.rfft(A) * self.dt
        E_f = np.fft.rfft(E) * self.dt
        T_f = np.fft.rfft(T) * self.dt

        if f_series is None or (f_series.shape == f_fft.shape and np.allclose(f_series, f_fft)):
            return f_fft, A_f, E_f, T_f
        # 插值到指定频率
        def interp_complex(y):
            return np.interp(f_series, f_fft, y.real) + 1j*np.interp(f_series, f_fft, y.imag)
        return f_series, interp_complex(A_f), interp_complex(E_f), interp_complex(T_f)

# ====== 适配你原函数接口 ======
adapter = EMRIAETAdapter(wf, dt=20.0)

def waveform(para_value, Tobs, frequency=None):
    # para_value 是参数向量 → 写回 wf
    for name, val in zip(PARAMS, para_value):
        setattr(adapter, name, val)
    f, A, E, T = adapter.get_fd_AET(frequency)
    idx = np.arange(len(f))  # 这里可以不使用
    return f, A, E, T, idx

def TQ_PSD(frequency):
    SAE, ST = TianQinNoise().noise_AET(frequency)
    return SAE, ST

def Innprod_wave(func1, func2, Snf, delta_f):
    return 4*(np.sum(func1.conjugate() * func2 / Snf * delta_f)).real

def AET_Innprod_wave(signalA, signalE, signalT, SAE, ST, frequency):
    delta_f = np.gradient(frequency)
    hh_A = Innprod_wave(signalA, signalA, SAE, delta_f)
    hh_E = Innprod_wave(signalE, signalE, SAE, delta_f)
    hh_T = Innprod_wave(signalT, signalT, ST,  delta_f)
    return hh_A + hh_E + hh_T

def simp_diff(para_value, delta, gap, Tobs, frequency=None):
    para_plus = para_value + delta
    para_subt = para_value - delta
    f, A_plus, E_plus, T_plus, _ = waveform(para_plus, Tobs, frequency)
    f, A_subt, E_subt, T_subt, _ = waveform(para_subt, Tobs, frequency)
    dA = (A_plus - A_subt)/gap/2
    dE = (E_plus - E_subt)/gap/2
    dT = (T_plus - T_subt)/gap/2
    return dA, dE, dT

def Compute_SNR_FIM(para_value, gap_ini=None, frequency=None, Tobs=5*YRSID_SI):
    Np = len(para_value)
    if gap_ini is None:
        gap_ini = np.ones(Np)*1e-6

    if frequency is None:
        frequency = np.arange(1e-4, 1e-2, 1/Tobs)

    SAE, ST = TQ_PSD(frequency)

    f, A, E, T, _ = waveform(para_value, Tobs, frequency)
    snr = np.sqrt(AET_Innprod_wave(A, E, T, SAE, ST, frequency))

    dA = np.zeros([Np, len(f)], dtype=np.complex128)
    dE = np.zeros_like(dA)
    dT = np.zeros_like(dA)
    for dim in range(Np):
        delta = np.zeros(Np); delta[dim] = gap_ini[dim]
        dA[dim], dE[dim], dT[dim] = simp_diff(para_value, delta, gap_ini[dim], Tobs, frequency)

    fim = np.zeros((Np, Np))
    delta_f = np.gradient(frequency)
    for i in range(Np):
        for j in range(Np):
            fim_A = Innprod_wave(dA[i], dA[j], SAE, delta_f)
            fim_E = Innprod_wave(dE[i], dE[j], SAE, delta_f)
            fim_T = Innprod_wave(dT[i], dT[j], ST,  delta_f)
            fim[i, j] = fim_A + fim_E + fim_T

    return snr, fim

# ====== 参数列表 & 真值向量 ======
PARAMS = ["M","mu","a","p0","e0","x0","qS","phiS","qK","phiK","dist",
          "Phi_phi0","Phi_theta0","Phi_r0"]

para_value = np.array([EMRIpars[p] for p in PARAMS])

# ====== 调用并输出 ======
snr, fim = Compute_SNR_FIM(para_value, Tobs=EMRIpars["T_obs"])
print("SNR:", snr)
print("FIM shape:", fim.shape)



: 

In [ ]:
import numpy as np
from gwspace.Waveform import EMRIWaveform
from gwspace.response import get_AET_td
from gwspace.Noise import TianQinNoise
from gwspace.constants import YRSID_SI
from gwspace.fishertool import fisher_matrix

EMRIpars = dict(
    M=4.3e6, mu=40.0, a=0.1, p0=9.0, e0=0.0, x0=1.0,
    qS=1.66, phiS=4.71, qK=0.2, phiK=0.2,
    dist=8.0e-6,
    Phi_phi0=1.0, Phi_theta0=2.0, Phi_r0=3.0,
    T_obs=5*YRSID_SI
)

wf = EMRIWaveform(**EMRIpars)
class EMRIAETAdapter:
    _local = {"wf", "dt", "det", "TDIgen"}

    def __init__(self, wf, dt, det="TQ", TDIgen=1):
        object.__setattr__(self, "wf", wf)
        object.__setattr__(self, "dt", dt)
        object.__setattr__(self, "det", det)
        object.__setattr__(self, "TDIgen", TDIgen)

    def __getattr__(self, name):
        return getattr(self.wf, name)

    def __setattr__(self, name, value):
        if name in self._local:
            object.__setattr__(self, name, value)
        else:
            setattr(self.wf, name, value)

    def get_tdi_response(self, f_series=None, channel="AET", det=None, TDIgen=None, **kwargs):
        if channel.upper() != "AET":
            raise ValueError("EMRIAETAdapter only supports AET.")

        det = det or self.det
        TDIgen = TDIgen or self.TDIgen

        t = np.arange(0, self.wf.T_obs, self.dt)
        A, E, T = get_AET_td(self.wf, t, det=det, TDIgen=TDIgen)

        N = len(t)
        f_fft = np.fft.rfftfreq(N, self.dt)
        A_f = np.fft.rfft(A) * self.dt
        E_f = np.fft.rfft(E) * self.dt
        T_f = np.fft.rfft(T) * self.dt

        h = np.vstack([A_f, E_f, T_f])

        if f_series is None:
            return h

        f_series = np.asarray(f_series)
        if f_series.shape == f_fft.shape and np.allclose(f_series, f_fft):
            return h

        def interp_complex(y):
            return np.interp(f_series, f_fft, y.real) + 1j*np.interp(f_series, f_fft, y.imag)

        return np.vstack([interp_complex(A_f), interp_complex(E_f), interp_complex(T_f)])
dt = 100.0  # 可以先粗一点；需要更精确时减小
adapter = EMRIAETAdapter(wf, dt=dt, det="TQ", TDIgen=1)

t = np.arange(0, wf.T_obs, dt)
f_series = np.fft.rfftfreq(len(t), dt)

params = [
    "M","mu","a","p0","e0","x0",
    "qS","phiS","qK","phiK","dist",
    "Phi_phi0","Phi_theta0","Phi_r0"
]

res = fisher_matrix(
    adapter,
    params=params,
    det="TQ",
    channel="AET",
    TDIgen=1,
    f_series=f_series,
    noise=TianQinNoise(),
    use_T=False,      # 如果想包括 T 通道改成 True
    rel_step=1e-6
)

print("SNR:", res["snr"])
print("sigma:", res["sigma"])


In [3]:
import numpy as np
from gwspace.Waveform import EMRIWaveform
from gwspace.response import get_AET_td
from gwspace.Noise import TianQinNoise
from gwspace.constants import YRSID_SI

# ====== EMRI 参数 ======
EMRIpars = dict(
    M=4.3e6, mu=40.0, a=0.1, p0=9.0, e0=0.0, x0=1.0,
    qS=1.66, phiS=4.71, qK=0.2, phiK=0.2,
    dist=8.0e-6,
    Phi_phi0=1.0, Phi_theta0=2.0, Phi_r0=3.0,
    T_obs=5*YRSID_SI
)
wf = EMRIWaveform(**EMRIpars)

# 参数顺序必须与 para_value 一致
PARAMS = [
    "M","mu","a","p0","e0","x0",
    "qS","phiS","qK","phiK","dist",
    "Phi_phi0","Phi_theta0","Phi_r0"
]

# 初值向量
para_value = np.array([EMRIpars[k] for k in PARAMS], dtype=float)

# FFT 时域步长（可调：大 → 快；小 → 精）
dt = 40.0

def waveform(para_value, Tobs, frequency=None):
    # 写回 EMRIWaveform 参数
    for name, val in zip(PARAMS, para_value):
        setattr(wf, name, val)

    # 时域 AET
    t = np.arange(0, Tobs, dt)
    A, E, T = get_AET_td(wf, t, det="TQ", TDIgen=1)

    # FFT → 频域
    N = len(t)
    f_fft = np.fft.rfftfreq(N, dt)
    A_f = np.fft.rfft(A) * dt
    E_f = np.fft.rfft(E) * dt
    T_f = np.fft.rfft(T) * dt

    if frequency is None:
        f = f_fft
        A, E, T = A_f, E_f, T_f
    else:
        f = np.asarray(frequency)
        if f[0] < f_fft[0] or f[-1] > f_fft[-1]:
            raise ValueError("frequency 超出 FFT 频率范围，请用 f_fft 或缩小频率范围。")

        def interp_complex(y):
            return np.interp(f, f_fft, y.real) + 1j*np.interp(f, f_fft, y.imag)

        A, E, T = interp_complex(A_f), interp_complex(E_f), interp_complex(T_f)

    idx = np.arange(len(f))
    return f, A, E, T, idx

def TQ_PSD(frequency):
    SAE, ST = TianQinNoise().noise_AET(frequency)
    return SAE, ST

# ====== 下面保留你原来的函数即可 ======
def check_para(para_value,gap,i_dim):
    # 如果你用 gap_choice，请按 EMRI 参数自行调整 min/max
    return gap

def Innprod_wave(func1,func2,Snf,delta_f):
    return 4*(np.sum(func1.conjugate() * func2 / Snf * delta_f)).real

def AET_Innprod_wave(signalA,signalE,signalT,SAE,ST,frequency):
    delta_f = np.gradient(frequency)
    hh_A = Innprod_wave(signalA,signalA,SAE,delta_f)
    hh_E = Innprod_wave(signalE,signalE,SAE,delta_f)
    hh_T = Innprod_wave(signalT,signalT,ST ,delta_f)
    return hh_A + hh_E + hh_T

def simp_diff(para_value, delta, gap, Tobs, frequency=None):
    para_plus = para_value+delta
    para_subt = para_value-delta
    f, A_plus, E_plus, T_plus, i = waveform(para_plus,Tobs,frequency)
    f, A_subt, E_subt, T_subt, i = waveform(para_subt,Tobs,frequency)
    dA = (A_plus - A_subt)/gap/2
    dE = (E_plus - E_subt)/gap/2
    dT = (T_plus - T_subt)/gap/2
    return dA, dE, dT

def Compute_SNR_FIM(para_value, gap_ini=None, gap_Adaptive=False, frequency=None, Tobs=5*YRSID_SI):
    Np = len(para_value)
    if gap_ini is None:
        gap_ini = np.ones(Np)*1e-6

    # 默认频率和 FFT 对齐（更稳）
    if frequency is None:
        t = np.arange(0, Tobs, dt)
        frequency = np.fft.rfftfreq(len(t), dt)

    SAE, ST = TQ_PSD(frequency)
    f, signalA, signalE, signalT, idx = waveform(para_value,Tobs,frequency)
    hh = AET_Innprod_wave(signalA,signalE,signalT,SAE,ST,frequency)
    snr  = np.sqrt(hh)

    gap = gap_ini
    dA = np.zeros([Np,len(f)],dtype=np.complex128)
    dE = np.zeros_like(dA)
    dT = np.zeros_like(dA)
    for dim in range(Np):
        delta = np.zeros(Np)
        delta[dim] = gap[dim]
        dA[dim,:], dE[dim,:], dT[dim,:] = simp_diff(para_value, delta, gap[dim], Tobs, frequency)

    fim = np.zeros((Np, Np))
    delta_f = np.gradient(frequency)
    for i in range(Np):
        for j in range(Np):
            fim_A = Innprod_wave(dA[i,:],dA[j,:],SAE,delta_f)
            fim_E = Innprod_wave(dE[i,:],dE[j,:],SAE,delta_f)
            fim_T = Innprod_wave(dT[i,:],dT[j,:],ST ,delta_f)
            fim[i,j] = fim_A + fim_E + fim_T
    return snr, fim

# snr, fim = Compute_SNR_FIM(para_value, Tobs=EMRIpars["T_obs"])
# print("SNR:", snr)
# print("FIM shape:", fim.shape)


In [2]:
import numpy as np
from gwspace.Waveform import EMRIWaveform
from gwspace.response import get_AET_td
from gwspace.Noise import TianQinNoise
from gwspace.constants import YRSID_SI

# ====== 1. EMRI 参数定义 ======
# 注意：dist=8.0e-6 取决于你的单位定义(Gpc?), 请确保该数值符合物理预期
EMRIpars = dict(
    M=4.3e6, mu=40.0, a=0.1, p0=9.0, e0=1e-4, x0=0.99,
    qS=1.66, phiS=4.71, qK=0.2, phiK=0.2,
    dist=8.0e-6, 
    Phi_phi0=1.0, Phi_theta0=2.0, Phi_r0=3.0,
    T_obs=0.1*YRSID_SI
)

# 实例化波形对象
wf = EMRIWaveform(**EMRIpars)

# 参数名称列表 (顺序必须固定)
PARAMS = [
    "M", "mu", "a", "p0", "e0", "x0",
    "qS", "phiS", "qK", "phiK", "dist",
    "Phi_phi0", "Phi_theta0", "Phi_r0"
]

# 提取参数初值
para_value = np.array([EMRIpars[k] for k in PARAMS], dtype=float)

# FFT 时域步长 (建议根据最高频率需求调整，dt=40s 对应 f_max=0.0125Hz)
dt = 40.0 

# ====== 2. 核心波形生成函数 ======
def waveform(para_value, Tobs, frequency=None):
    """
    根据参数生成频域 AET 波形
    """
    # 1. 更新对象属性
    for name, val in zip(PARAMS, para_value):
        setattr(wf, name, val)

    # 2. 生成时域 AET
    t = np.arange(0, Tobs, dt)
    # 注意：TDIgen=1 或 2 取决于 gwspace 版本要求，通常 2nd generation TDI 更准
    A, E, T = get_AET_td(wf, t, det="TQ", TDIgen=1) 

    # 3. FFT 转频域
    N = len(t)
    f_fft = np.fft.rfftfreq(N, dt)
    # 注意：乘 dt 是为了近似连续傅里叶变换
    A_f = np.fft.rfft(A) * dt
    E_f = np.fft.rfft(E) * dt
    T_f = np.fft.rfft(T) * dt

    # 4. 频率插值 (如果指定了 frequency)
    if frequency is None:
        f = f_fft
        A_out, E_out, T_out = A_f, E_f, T_f
    else:
        f = np.asarray(frequency)
        # 边界检查
        if f[0] < f_fft[0] or f[-1] > f_fft[-1]:
            # 这里的警告比报错更温和，防止浮点误差导致的边界溢出
            print("Warning: 请求频率超出 FFT 范围，可能导致插值误差。")
        
        # 定义复数插值函数
        def interp_complex(y_data):
            return np.interp(f, f_fft, y_data.real) + 1j * np.interp(f, f_fft, y_data.imag)

        A_out = interp_complex(A_f)
        E_out = interp_complex(E_f)
        T_out = interp_complex(T_f)

    idx = np.arange(len(f))
    return f, A_out, E_out, T_out, idx

# ====== 3. 噪声谱函数 ======
def TQ_PSD(frequency):
    # 使用 gwspace 自带的天琴噪声模型
    # 注意：gwspace 的 noise_AET 返回可能是 PSD 或 ASD，确认是 PSD (Power)
    SAE, ST = TianQinNoise().noise_AET(frequency)
    return SAE, ST

# ====== 4. 内积与微分工具 ======
def Innprod_wave(func1, func2, Snf, delta_f):
    """
    计算内积 <a|b> = 4 * Re( sum( a* b / Sn ) * df )
    """
    # 加上 1e-30 防止 Snf 为 0 的除零错误
    vals = func1.conjugate() * func2 / (Snf + 1e-30) 
    return 4 * np.real(np.sum(vals * delta_f))

def AET_Innprod_wave(signalA, signalE, signalT, SAE, ST, frequency):
    """
    计算 A, E, T 三通道总信噪比平方
    """
    delta_f = np.gradient(frequency)
    # 假设 A 和 E 通道噪声相同 SAE，T 通道噪声 ST
    hh_A = Innprod_wave(signalA, signalA, SAE, delta_f)
    hh_E = Innprod_wave(signalE, signalE, SAE, delta_f)
    hh_T = Innprod_wave(signalT, signalT, ST, delta_f)
    return hh_A + hh_E + hh_T

def simp_diff(para_value, delta, gap, Tobs, frequency=None):
    """
    中心差分法求导
    gap: 标量，步长值
    delta: 向量，仅在当前求导维度有值，其余为0
    """
    # 右微扰
    para_plus = para_value + delta
    _, A_plus, E_plus, T_plus, _ = waveform(para_plus, Tobs, frequency)
    
    # 左微扰
    para_subt = para_value - delta
    _, A_subt, E_subt, T_subt, _ = waveform(para_subt, Tobs, frequency)
    
    # 计算导数 (dy/dx)
    dA = (A_plus - A_subt) / (2 * gap)
    dE = (E_plus - E_subt) / (2 * gap)
    dT = (T_plus - T_subt) / (2 * gap)
    
    return dA, dE, dT

# ====== 5. 主计算函数 (含修正后的步长逻辑) ======
def Compute_SNR_FIM(para_value, gap_ini=None, frequency=None, Tobs=5*YRSID_SI):
    Np = len(para_value)
    
    # --- [修正点] 智能步长生成逻辑 ---
    if gap_ini is None:
        gap_ini = np.zeros(Np)
        for i, name in enumerate(PARAMS):
            val = para_value[i]
            # 针对大数值参数 (M, mu, dist) 使用相对步长
            if name in ['M', 'mu', 'dist']:
                gap_ini[i] = np.abs(val) * 1e-5
            # 针对可能为0的参数 (e0, a, x0, p0) 使用绝对步长，避免乘0
            elif name in ['a', 'p0', 'e0', 'x0']:
                gap_ini[i] = 1e-5
            # 针对角度参数 (Radians)
            else:
                gap_ini[i] = 1e-6
        print(f"Auto-generated step sizes (gap): \n{gap_ini}")
    
    # 频率网格设置
    if frequency is None:
        t_temp = np.arange(0, Tobs, dt)
        frequency = np.fft.rfftfreq(len(t_temp), dt)
        # 去掉直流分量 f=0，防止噪声无穷大
        if frequency[0] == 0:
            frequency[0] = frequency[1] * 1e-3 # 或者直接切片 frequency = frequency[1:]

    # 1. 计算中心值的 SNR
    SAE, ST = TQ_PSD(frequency)
    _, sA, sE, sT, _ = waveform(para_value, Tobs, frequency)
    hh = AET_Innprod_wave(sA, sE, sT, SAE, ST, frequency)
    snr = np.sqrt(hh)
    
    # 2. 计算导数矩阵
    # 形状: [参数个数, 频率点数]
    dA_list = []
    dE_list = []
    dT_list = []
    
    print("Calculating derivatives...")
    for dim in range(Np):
        # 构造微扰向量
        delta_vec = np.zeros(Np)
        current_gap = gap_ini[dim]
        delta_vec[dim] = current_gap
        
        # 计算该维度的导数
        dA, dE, dT = simp_diff(para_value, delta_vec, current_gap, Tobs, frequency)
        dA_list.append(dA)
        dE_list.append(dE)
        dT_list.append(dT)
    
    # 3. 组装 Fisher 矩阵
    print("Computing Fisher Matrix...")
    fim = np.zeros((Np, Np))
    delta_f = np.gradient(frequency)
    
    for i in range(Np):
        for j in range(i, Np): # 利用对称性，只算上三角
            # A通道贡献
            f_A = Innprod_wave(dA_list[i], dA_list[j], SAE, delta_f)
            # E通道贡献
            f_E = Innprod_wave(dE_list[i], dE_list[j], SAE, delta_f)
            # T通道贡献
            f_T = Innprod_wave(dT_list[i], dT_list[j], ST,  delta_f)
            
            val = f_A + f_E + f_T
            fim[i, j] = val
            fim[j, i] = val # 对称赋值
            
    return snr, fim

#====== 6. 运行示例 ======
if __name__ == "__main__":
    # 调用计算
    print("Starting calculation...")
    snr_val, fim_mat = Compute_SNR_FIM(para_value, Tobs=EMRIpars["T_obs"])
    
    print(f"\nResult SNR: {snr_val:.4f}")
    
    # 计算误差 (Cramer-Rao Bound: sigma = sqrt(diagonal(inverse(FIM))))
    try:
        cov = np.linalg.inv(fim_mat)
        errors = np.sqrt(np.diag(cov))
        print("\nParameter Errors (1-sigma):")
        for i, name in enumerate(PARAMS):
            print(f"{name}: {para_value[i]:.3e} +/- {errors[i]:.3e}")
    except np.linalg.LinAlgError:
        print("\nFIM is singular, cannot invert. (可能是参数退化或步长设置不当)")
# import time
# t0 = time.perf_counter()
# _ = waveform(para_value, EMRIpars["T_obs"], frequency=None)
# print("single waveform:", time.perf_counter() - t0, "s")

Starting calculation...
Auto-generated step sizes (gap): 
[4.3e+01 4.0e-04 1.0e-05 1.0e-05 1.0e-05 1.0e-05 1.0e-06 1.0e-06 1.0e-06
 1.0e-06 8.0e-11 1.0e-06 1.0e-06 1.0e-06]
Calculating derivatives...
Computing Fisher Matrix...

Result SNR: 0.0000

FIM is singular, cannot invert. (可能是参数退化或步长设置不当)


In [ ]:
import numpy as np
from gwspace.Waveform import EMRIWaveform
from gwspace.response import get_AET_td
from gwspace.Noise import TianQinNoise
from gwspace.constants import YRSID_SI

# ====== 1. EMRI 参数定义 ======
# 重要：e0/x0/p0 远离边界，避免中心差分越界/失真
EMRIpars = dict(
    M=4.3e6, mu=40.0, a=0.1, p0=10.0, e0=1e-3, x0=0.9,
    qS=1.66, phiS=4.71, qK=0.2, phiK=0.2,
    dist=8.0e-6,
    Phi_phi0=1.0, Phi_theta0=2.0, Phi_r0=3.0,
    T_obs=0.1*YRSID_SI
)

# 实例化波形对象
wf = EMRIWaveform(**EMRIpars)

# 参数名称列表 
PARAMS = [
    "M", "mu", "a", "p0", "e0", "x0",
    "qS", "phiS", "qK", "phiK", "dist",
    "Phi_phi0", "Phi_theta0", "Phi_r0"
]

# 提取参数初值
para_value = np.array([EMRIpars[k] for k in PARAMS], dtype=float)

# FFT 时域步长
dt = 40.0

# ====== 2. 核心波形生成函数 ======
def waveform(para_value, Tobs, frequency=None):
    """
    根据参数生成频域 AET 波形
    """
    # 1. 更新对象属性
    for name, val in zip(PARAMS, para_value):
        setattr(wf, name, val)

    # 2. 生成时域 AET
    t = np.arange(0, Tobs, dt)
    A, E, T = get_AET_td(wf, t, det="TQ", TDIgen=1)

    # 3. FFT 转频域
    N = len(t)
    f_fft = np.fft.rfftfreq(N, dt)
    A_f = np.fft.rfft(A) * dt
    E_f = np.fft.rfft(E) * dt
    T_f = np.fft.rfft(T) * dt

    # 4. 频率插值 (如果指定了 frequency)
    if frequency is None:
        f = f_fft
        A_out, E_out, T_out = A_f, E_f, T_f
    else:
        f = np.asarray(frequency)
        if f[0] < f_fft[0] or f[-1] > f_fft[-1]:
            print("Warning: 请求频率超出 FFT 范围，可能导致插值误差。")

        def interp_complex(y_data):
            return np.interp(f, f_fft, y_data.real) + 1j * np.interp(f, f_fft, y_data.imag)

        A_out = interp_complex(A_f)
        E_out = interp_complex(E_f)
        T_out = interp_complex(T_f)

    idx = np.arange(len(f))
    return f, A_out, E_out, T_out, idx

# ====== 3. 噪声谱函数 ======
def TQ_PSD(frequency):
    SAE, ST = TianQinNoise().noise_AET(frequency)
    return SAE, ST

# ====== 4. 内积与微分工具 ======
def Innprod_wave(func1, func2, Snf, df):
    """
    计算内积 <a|b> = 4 * Re( sum( a* b / Sn ) * df )
    """
    vals = func1.conjugate() * func2 / (Snf + 1e-30)
    return 4 * np.real(np.sum(vals) * df)

def AET_Innprod_wave(signalA, signalE, signalT, SAE, ST, df):
    """
    计算 A, E, T 三通道总信噪比平方
    """
    hh_A = Innprod_wave(signalA, signalA, SAE, df)
    hh_E = Innprod_wave(signalE, signalE, SAE, df)
    hh_T = Innprod_wave(signalT, signalT, ST, df)
    return hh_A + hh_E + hh_T

def simp_diff(para_value, delta, gap, Tobs, frequency=None):
    """
    中心差分法求导
    gap: 标量，步长值
    delta: 向量，仅在当前求导维度有值，其余为0
    """
    para_plus = para_value + delta
    _, A_plus, E_plus, T_plus, _ = waveform(para_plus, Tobs, frequency)

    para_subt = para_value - delta
    _, A_subt, E_subt, T_subt, _ = waveform(para_subt, Tobs, frequency)

    dA = (A_plus - A_subt) / (2 * gap)
    dE = (E_plus - E_subt) / (2 * gap)
    dT = (T_plus - T_subt) / (2 * gap)
    return dA, dE, dT

# ====== 5. 主计算函数 ======
def Compute_SNR_FIM(para_value, gap_ini=None, frequency=None, Tobs=5*YRSID_SI):
    Np = len(para_value)

    # 智能步长生成逻辑
    if gap_ini is None:
        gap_ini = np.zeros(Np)
        for i, name in enumerate(PARAMS):
            val = para_value[i]
            if name in ['M', 'mu', 'dist']:
                gap_ini[i] = np.abs(val) * 1e-5
            elif name in ['a', 'p0', 'e0', 'x0']:
                gap_ini[i] = 1e-5
            else:
                gap_ini[i] = 1e-6
        print(f"Auto-generated step sizes (gap): \n{gap_ini}")

    # 频率网格设置（关键修正：删掉 f=0）
    if frequency is None:
        t_temp = np.arange(0, Tobs, dt)
        frequency = np.fft.rfftfreq(len(t_temp), dt)[1:]  # 直接丢掉 f=0

    # 常数 df
    df = frequency[1] - frequency[0]

    # 1. 计算中心值的 SNR
    SAE, ST = TQ_PSD(frequency)
    _, sA, sE, sT, _ = waveform(para_value, Tobs, frequency)
    hh = AET_Innprod_wave(sA, sE, sT, SAE, ST, df)
    snr = np.sqrt(hh)

    # 2. 计算导数矩阵
    dA_list = []
    dE_list = []
    dT_list = []

    print("Calculating derivatives...")
    for dim in range(Np):
        delta_vec = np.zeros(Np)
        current_gap = gap_ini[dim]
        delta_vec[dim] = current_gap

        dA, dE, dT = simp_diff(para_value, delta_vec, current_gap, Tobs, frequency)
        dA_list.append(dA)
        dE_list.append(dE)
        dT_list.append(dT)

    # 3. 组装 Fisher 矩阵
    print("Computing Fisher Matrix...")
    fim = np.zeros((Np, Np))

    for i in range(Np):
        for j in range(i, Np):
            f_A = Innprod_wave(dA_list[i], dA_list[j], SAE, df)
            f_E = Innprod_wave(dE_list[i], dE_list[j], SAE, df)
            f_T = Innprod_wave(dT_list[i], dT_list[j], ST, df)

            val = f_A + f_E + f_T
            fim[i, j] = val
            fim[j, i] = val

    return snr, fim

# ====== 6. 运行 ======
if __name__ == "__main__":
    print("Starting calculation...")
    snr_val, fim_mat = Compute_SNR_FIM(para_value, Tobs=EMRIpars["T_obs"])

    print(f"\nResult SNR: {snr_val:.6e}")

    try:
        cov = np.linalg.inv(fim_mat)
        errors = np.sqrt(np.diag(cov))
        print("\nParameter Errors (1-sigma):")
        for i, name in enumerate(PARAMS):
            print(f"{name}: {para_value[i]:.3e} +/- {errors[i]:.3e}")
    except np.linalg.LinAlgError:
        print("\nFIM is singular, cannot invert. (可能是参数退化或步长设置不当)")


Starting calculation...
Auto-generated step sizes (gap): 
[4.3e+01 4.0e-04 1.0e-05 1.0e-05 1.0e-05 1.0e-05 1.0e-06 1.0e-06 1.0e-06
 1.0e-06 8.0e-11 1.0e-06 1.0e-06 1.0e-06]
Calculating derivatives...
Computing Fisher Matrix...

Result SNR: 2.199959e-09

FIM is singular, cannot invert. (可能是参数退化或步长设置不当)
